In [0]:
%python
# ====================================================================
# DATA PROFILING - BRONZE LAYER
# ====================================================================
# Purpose: Explore and understand the bronze tables before cleaning
# Output: Data quality insights, null counts, duplicates, distributions
# ====================================================================

from pyspark.sql.functions import (
    col, count, countDistinct, min, max, avg, sum, when,
    isnan, isnull, length, approx_count_distinct
)
from pyspark.sql.types import StringType, NumericType, TimestampType, DateType

# Configuration
PROJECT_NAME = "retail"
CATALOG = "workspace"
BRONZE_SCHEMA = f"{PROJECT_NAME}_bronze"

print("=" * 80)
print("🔍 DATA PROFILING - BRONZE LAYER")
print("=" * 80)
print(f"Catalog: {CATALOG}")
print(f"Schema: {BRONZE_SCHEMA}")
print("=" * 80 + "\n")

In [0]:
%python
# ====================================================================
# DATA PROFILING - BRONZE LAYER
# ====================================================================
# Purpose: Explore and understand the bronze tables before cleaning
# Output: Data quality insights, null counts, duplicates, distributions
# ====================================================================

from pyspark.sql.functions import (
    col, count, countDistinct, min, max, avg, sum, when,
    isnan, isnull, length, approx_count_distinct
)
from pyspark.sql.types import StringType, NumericType, TimestampType, DateType

# Configuration
PROJECT_NAME = "retail"
CATALOG = "workspace"
BRONZE_SCHEMA = f"{PROJECT_NAME}_bronze"

print("=" * 80)
print("🔍 DATA PROFILING - BRONZE LAYER")
print("=" * 80)
print(f"Catalog: {CATALOG}")
print(f"Schema: {BRONZE_SCHEMA}")
print("=" * 80 + "\n")

In [0]:
%python
# ====================================================================
# DATA PROFILING - BRONZE LAYER
# ====================================================================
# Purpose: Explore and understand the bronze tables before cleaning
# Output: Data quality insights, null counts, duplicates, distributions
# ====================================================================

from pyspark.sql.functions import (
    col, count, countDistinct, min, max, avg, sum, when,
    isnan, isnull, length, approx_count_distinct
)
from pyspark.sql.types import StringType, NumericType, TimestampType, DateType

# Configuration
PROJECT_NAME = "retail"
CATALOG = "workspace"
BRONZE_SCHEMA = f"{PROJECT_NAME}_bronze"

print("=" * 80)
print("🔍 DATA PROFILING - BRONZE LAYER")
print("=" * 80)
print(f"Catalog: {CATALOG}")
print(f"Schema: {BRONZE_SCHEMA}")
print("=" * 80 + "\n")

In [0]:
%python
# ====================================================================
# PROFILING HELPER FUNCTIONS
# ====================================================================

def get_table_summary(table_name):
    """Get basic table statistics."""
    full_table_name = f"{CATALOG}.{BRONZE_SCHEMA}.{table_name}"
    df = spark.table(full_table_name)
    
    row_count = df.count()
    col_count = len(df.columns)
    
    print(f"\n{'='*80}")
    print(f"📊 TABLE: {table_name}")
    print(f"{'='*80}")
    print(f"Total Rows: {row_count:,}")
    print(f"Total Columns: {col_count}")
    print(f"{'-'*80}\n")
    
    return df, row_count


def check_data_types(df):
    """Display data types for all columns."""
    print("📋 DATA TYPES:")
    print(f"{'Column Name':<40} {'Data Type':<20}")
    print("-" * 80)
    
    for field in df.schema.fields:
        print(f"{field.name:<40} {str(field.dataType):<20}")
    print()


def check_nulls(df):
    """Count nulls and null percentage for each column."""
    total_rows = df.count()
    
    print("🔍 NULL VALUE ANALYSIS:")
    print(f"{'Column Name':<40} {'Null Count':>15} {'Null %':>10}")
    print("-" * 80)
    
    null_counts = []
    
    for column in df.columns:
        null_count = df.filter(col(column).isNull()).count()
        null_pct = (null_count / total_rows * 100) if total_rows > 0 else 0
        null_counts.append((column, null_count, null_pct))
        
        # Highlight columns with high null percentage
        if null_pct > 50:
            indicator = "⚠️"
        elif null_pct > 10:
            indicator = "⚡"
        else:
            indicator = "✅"
        
        print(f"{column:<40} {null_count:>15,} {null_pct:>9.2f}% {indicator}")
    
    print()
    return null_counts


def check_duplicates(df, key_columns):
    """Check for duplicate records based on key columns."""
    total_rows = df.count()
    distinct_rows = df.select(key_columns).distinct().count()
    duplicates = total_rows - distinct_rows
    
    print("🔄 DUPLICATE CHECK:")
    print(f"Key Column(s): {', '.join(key_columns)}")
    print(f"Total Rows: {total_rows:,}")
    print(f"Distinct Rows: {distinct_rows:,}")
    print(f"Duplicates: {duplicates:,}")
    
    if duplicates > 0:
        print("⚠️ DUPLICATES FOUND!")
    else:
        print("✅ No duplicates")
    print()


def analyze_numeric_columns(df):
    """Get min, max, avg for numeric columns."""
    numeric_cols = [field.name for field in df.schema.fields 
                    if isinstance(field.dataType, NumericType)]
    
    if not numeric_cols:
        print("No numeric columns found.\n")
        return
    
    print("📊 NUMERIC COLUMN STATISTICS:")
    print(f"{'Column Name':<40} {'Min':>12} {'Max':>12} {'Avg':>12}")
    print("-" * 80)
    
    for col_name in numeric_cols:
        stats = df.select(
            min(col(col_name)).alias('min_val'),
            max(col(col_name)).alias('max_val'),
            avg(col(col_name)).alias('avg_val')
        ).first()
        
        min_val = stats['min_val'] if stats['min_val'] is not None else 'NULL'
        max_val = stats['max_val'] if stats['max_val'] is not None else 'NULL'
        avg_val = f"{stats['avg_val']:.2f}" if stats['avg_val'] is not None else 'NULL'
        
        print(f"{col_name:<40} {str(min_val):>12} {str(max_val):>12} {str(avg_val):>12}")
    print()


def analyze_categorical_columns(df, max_display=10):
    """Show unique value counts for string columns."""
    string_cols = [field.name for field in df.schema.fields 
                   if isinstance(field.dataType, StringType)]
    
    if not string_cols:
        print("No string columns found.\n")
        return
    
    print("📝 CATEGORICAL COLUMN ANALYSIS:")
    print(f"{'Column Name':<40} {'Unique Values':>15}")
    print("-" * 80)
    
    for col_name in string_cols:
        unique_count = df.select(col_name).distinct().count()
        print(f"{col_name:<40} {unique_count:>15,}")
    print()


def analyze_timestamp_columns(df):
    """Show date ranges for timestamp columns."""
    timestamp_cols = [field.name for field in df.schema.fields 
                      if isinstance(field.dataType, (TimestampType, DateType))]
    
    if not timestamp_cols:
        print("No timestamp columns found.\n")
        return
    
    print("📅 TIMESTAMP COLUMN ANALYSIS:")
    print(f"{'Column Name':<40} {'Earliest':>20} {'Latest':>20}")
    print("-" * 80)
    
    for col_name in timestamp_cols:
        # First parse as timestamp if it's a string
        df_temp = df.withColumn("temp_ts", col(col_name).cast("timestamp"))
        
        stats = df_temp.select(
            min("temp_ts").alias('min_date'),
            max("temp_ts").alias('max_date')
        ).first()
        
        min_date = str(stats['min_date']) if stats['min_date'] else 'NULL'
        max_date = str(stats['max_date']) if stats['max_date'] else 'NULL'
        
        print(f"{col_name:<40} {min_date:>20} {max_date:>20}")
    print()


def profile_table(table_name, key_columns):
    """Complete profiling workflow for a single table."""
    df, row_count = get_table_summary(table_name)
    check_data_types(df)
    check_nulls(df)
    check_duplicates(df, key_columns)
    analyze_numeric_columns(df)
    analyze_categorical_columns(df)
    analyze_timestamp_columns(df)
    
    print("=" * 80)
    print(f"✅ Profiling complete for {table_name}")
    print("=" * 80 + "\n\n")

In [0]:
%python
# ====================================================================
# PROFILE: ORDERS
# ====================================================================
profile_table("bronze_orders", ["order_id"])

In [0]:
%python
# ====================================================================
# PROFILE: CUSTOMERS
# ====================================================================
profile_table("bronze_customers", ["customer_id"])

In [0]:
%python
# ====================================================================
# PROFILE: ORDER ITEMS
# ====================================================================
profile_table("bronze_order_items", ["order_id", "order_item_id"])

In [0]:
%python
# ====================================================================
# PROFILE: PAYMENTS
# ====================================================================
profile_table("bronze_order_payments", ["order_id", "payment_sequential"])

In [0]:
%python
# ====================================================================
# PROFILE: REVIEWS
# ====================================================================
profile_table("bronze_order_reviews", ["review_id"])

In [0]:
%python
# ====================================================================
# PROFILE: PRODUCTS
# ====================================================================
profile_table("bronze_products", ["product_id"])

In [0]:
%python
# ====================================================================
# PROFILE: SELLERS
# ====================================================================
profile_table("bronze_sellers", ["seller_id"])

In [0]:
%python
# ====================================================================
# PROFILE: GEOLOCATION
# ====================================================================
profile_table("bronze_geolocation", ["geolocation_zip_code_prefix"])

In [0]:
%python
# ====================================================================
# PROFILE: CATEGORY TRANSLATION
# ====================================================================
profile_table("bronze_category_translation", ["product_category_name"])

In [0]:
%python
# ====================================================================
# FOREIGN KEY VALIDATION
# ====================================================================
print("\n" + "="*80)
print("🔗 FOREIGN KEY VALIDATION")
print("="*80 + "\n")

# Check if all customer_ids in orders exist in customers table
orders_df = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.bronze_orders")
customers_df = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.bronze_customers")

total_orders = orders_df.count()
orders_with_valid_customers = orders_df.join(
    customers_df, 
    "customer_id", 
    "inner"
).count()

orphan_orders = total_orders - orders_with_valid_customers

print(f"📊 Orders → Customers Relationship:")
print(f"  Total Orders: {total_orders:,}")
print(f"  Orders with valid customer_id: {orders_with_valid_customers:,}")
print(f"  Orphan orders (no matching customer): {orphan_orders:,}")

if orphan_orders > 0:
    print("  ⚠️ WARNING: Orphan records found!")
else:
    print("  ✅ All orders have valid customers")

print()

# Check if all product_ids in order_items exist in products table
order_items_df = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.bronze_order_items")
products_df = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.bronze_products")

total_items = order_items_df.count()
items_with_valid_products = order_items_df.join(
    products_df,
    "product_id",
    "inner"
).count()

orphan_items = total_items - items_with_valid_products

print(f"📊 Order Items → Products Relationship:")
print(f"  Total Order Items: {total_items:,}")
print(f"  Items with valid product_id: {items_with_valid_products:,}")
print(f"  Orphan items (no matching product): {orphan_items:,}")

if orphan_items > 0:
    print("  ⚠️ WARNING: Orphan records found!")
else:
    print("  ✅ All order items have valid products")

print()

# Check if all seller_ids in order_items exist in sellers table
sellers_df = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.bronze_sellers")

items_with_valid_sellers = order_items_df.join(
    sellers_df,
    "seller_id",
    "inner"
).count()

orphan_seller_items = total_items - items_with_valid_sellers

print(f"📊 Order Items → Sellers Relationship:")
print(f"  Total Order Items: {total_items:,}")
print(f"  Items with valid seller_id: {items_with_valid_sellers:,}")
print(f"  Orphan items (no matching seller): {orphan_seller_items:,}")

if orphan_seller_items > 0:
    print("  ⚠️ WARNING: Orphan records found!")
else:
    print("  ✅ All order items have valid sellers")

print("\n" + "="*80 + "\n")

In [0]:
%python
# ====================================================================
# OVERALL SUMMARY
# ====================================================================
print("\n" + "="*80)
print("📈 BRONZE LAYER SUMMARY")
print("="*80 + "\n")

bronze_tables = [
    "bronze_orders",
    "bronze_customers",
    "bronze_order_items",
    "bronze_order_payments",
    "bronze_order_reviews",
    "bronze_products",
    "bronze_sellers",
    "bronze_geolocation",
    "bronze_category_translation"
]

print(f"{'Table Name':<40} {'Row Count':>15} {'Column Count':>15}")
print("-" * 80)

total_rows = 0
for table_name in bronze_tables:
    df = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.{table_name}")
    row_count = df.count()
    col_count = len(df.columns)
    total_rows += row_count
    
    print(f"{table_name:<40} {row_count:>15,} {col_count:>15}")

print("-" * 80)
print(f"{'TOTAL ROWS':<40} {total_rows:>15,}")
print("="*80)

print("\n✅ Data profiling complete!")
print("\n📝 Next Steps:")
print("  1. Review the profiling results above")
print("  2. Identify data quality issues (nulls, duplicates, invalid values)")
print("  3. Document your findings in sql/bronze_profiling/ folder")
print("  4. Run 01_clean_bronze_tables.py with your cleaning strategy")
print()